# plfrets — monthly Barra factor WLS

Lazy cross-sectional weighted least squares of `ret` on Barra style + industry
exposures, one regression per `date` (month).

**Input:** `./parquet_files/fexp_panel.parquet` (from `fexp_panel2_parquet.ipynb`)  
**Output:** monthly betas + fit statistics

- Weight: `regwt = 1 / srisk²`
- No intercept (`WORLD` ≡ 1 for all names)
- `country_gem4 == "USA"` filter is a pushdown placeholder on the lazy scan

In [ ]:
from pathlib import Path

import polars as pl
import polars_ols  # noqa: F401 — registers .least_squares namespace

PARQUET_PATH = Path("parquet_files/fexp_panel.parquet")
OUTPUT_PATH = Path("parquet_files/fexp_wls_betas.parquet")

RISK_FACTORS = [
    "WORLD", "BETA", "BTOP", "DIVYILD", "EARNQLTY", "EARNVAR", "EARNYILD",
    "GROWTH", "INVSQLTY", "LEVERAGE", "LIQUIDTY",
]

INDUSTRY_FACTORS = [
    "AEROSPCE", "AIRLINES", "DIVMETAL", "AUTOCOMP", "BANKS", "BIOTECH",
    "BLDCNSTR", "CHEMICAL", "COMMSVCS", "COMMUNIC", "COMPUTER", "CONSTPP",
    "CONSDUR", "CONSVCS", "DIVFIN", "ENERGY", "AGROCHEM", "FOODPRD", "FOODRETL",
    "GOLD", "HLTHEQP", "HLTHSVC", "HSHLDPRD", "INOILGAS", "INSURNCE", "INTERNET",
    "SOFTWARE", "MACHINRY", "MEDIA", "OILGAS", "OILEXPL", "PHARMA", "PRECMETL",
    "REALEST", "RETAIL", "SEMICOND", "SMICNDEQ", "STEEL", "TELECOM", "TRNSPORT",
    "UTILITY", "CAPMRKTS", "RGNLBNKS", "THRIFTS", "RLESTMNG",
]

FACTOR_COLUMNS = RISK_FACTORS + INDUSTRY_FACTORS
print(f"{len(RISK_FACTORS)} style + {len(INDUSTRY_FACTORS)} industry = {len(FACTOR_COLUMNS)} factors")

In [ ]:
def wls_expr(mode: str) -> pl.Expr:
    return pl.col("ret").least_squares.wls(
        *[pl.col(name) for name in FACTOR_COLUMNS],
        sample_weights=pl.col("regwt"),
        add_intercept=False,
        null_policy="drop",
        solve_method="svd",
        mode=mode,
    )


weighted_mean_ret = (pl.col("ret") * pl.col("regwt")).sum() / pl.col("regwt").sum()

lazy_plan = (
    pl.scan_parquet(PARQUET_PATH)
    .filter(pl.col("country_gem4") == "USA")  # pushdown placeholder
    .with_columns((1.0 / pl.col("srisk").pow(2)).alias("regwt"))
    .group_by("date")
    .agg(
        pl.len().alias("n_obs"),
        wls_expr("coefficients").alias("betas"),
        (pl.col("regwt") * wls_expr("residuals").pow(2)).sum().alias("sse"),
        (pl.col("regwt") * (pl.col("ret") - weighted_mean_ret).pow(2))
        .sum()
        .alias("tss"),
    )
    .with_columns((1.0 - pl.col("sse") / pl.col("tss")).alias("r2"))
    .sort("date")
)

lazy_plan.explain(optimized=True)

In [ ]:
results_df = lazy_plan.collect()

monthly_betas = results_df.unnest("betas").sort("date")
print(f"{monthly_betas.height:,} months × {len(FACTOR_COLUMNS)} betas")

summary_cols = ["date", "n_obs", "sse", "tss", "r2"]
monthly_betas.select(summary_cols).head(5)

In [ ]:
monthly_betas.write_parquet(OUTPUT_PATH)
print(f"Wrote {OUTPUT_PATH.resolve()}")

monthly_betas.select(["date", "WORLD", "BETA", "BANKS", "SOFTWARE", "r2"]).tail(5)